# Triton Мидтерм: fp16 → int4 quantization + packing into `uint8`

Этот ноутбук — **самый базовая рабочая верся** для пункта 1 проекта:

Реализовать кернель для квантизации 2D матрицы из fp16 в int4
и последующей упаковки квантизованной матрицы в int8 или int32.
При этом потребляемая память должна уменьшиться в 4 раза.

### Что здесь реализовано
- Triton kernel для вычисления `scale` по строкам (`absmax / 7`)
- Triton kernel для quantize + pack в `uint8`
- PyTorch reference для unpack / dequant для проверки



In [1]:

import math
import time

import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), "Нужна CUDA GPU для запуска Triton kernels"
device = "cuda"

print("torch:", torch.__version__)
print("triton:", triton.__version__)
print("cuda device:", torch.cuda.get_device_name(0))


torch: 2.5.1+cu121
triton: 3.1.0
cuda device: NVIDIA A100 80GB PCIe


In [2]:

@triton.jit
def row_absmax_kernel(
    x_ptr,
    scales_ptr,
    stride_xm,
    stride_xn,
    M,
    N,
    BLOCK_N: tl.constexpr,
):
    row = tl.program_id(0)
    offs_n = tl.arange(0, BLOCK_N)
    mask = offs_n < N

    x = tl.load(x_ptr + row * stride_xm + offs_n * stride_xn, mask=mask, other=0.0)
    absmax = tl.max(tl.abs(x), axis=0)
    scale = absmax / 7.0
    scale = tl.where(scale > 0, scale, 1.0)

    tl.store(scales_ptr + row, scale)


@triton.jit
def quant_pack_int4_kernel(
    x_ptr,
    scales_ptr,
    packed_ptr,
    stride_xm,
    stride_xn,
    stride_pm,
    stride_pn,
    M,
    N,
    BLOCK_PACKS: tl.constexpr,
):
    row = tl.program_id(0)
    pack_block = tl.program_id(1)

    offs_pack = pack_block * BLOCK_PACKS + tl.arange(0, BLOCK_PACKS)

    col0 = 2 * offs_pack
    col1 = col0 + 1

    mask0 = col0 < N
    mask1 = col1 < N

    x0 = tl.load(x_ptr + row * stride_xm + col0 * stride_xn, mask=mask0, other=0.0)
    x1 = tl.load(x_ptr + row * stride_xm + col1 * stride_xn, mask=mask1, other=0.0)

    scale = tl.load(scales_ptr + row)
    inv_scale = 1.0 / scale

    q0f = x0 * inv_scale
    q1f = x1 * inv_scale

    q0f = tl.where(q0f >= 0, q0f + 0.5, q0f - 0.5)
    q1f = tl.where(q1f >= 0, q1f + 0.5, q1f - 0.5)

    q0 = tl.cast(q0f, tl.int32)
    q1 = tl.cast(q1f, tl.int32)

    q0 = tl.maximum(tl.minimum(q0, 7), -8)
    q1 = tl.maximum(tl.minimum(q1, 7), -8)

    q0u = tl.cast(q0 + 8, tl.uint8)
    q1u = tl.cast(q1 + 8, tl.uint8)

    packed = q0u | (q1u << 4)

    out_mask = offs_pack < ((N + 1) // 2)
    tl.store(packed_ptr + row * stride_pm + offs_pack * stride_pn, packed, mask=out_mask)


def quantize_fp16_to_int4_packed_triton(x: torch.Tensor):
    """
    x: [M, N] torch.float16 / torch.bfloat16 on CUDA
    returns:
        packed: [M, ceil(N/2)] uint8
        scales: [M] float32
    """
    assert x.is_cuda
    assert x.ndim == 2
    assert x.dtype in (torch.float16, torch.bfloat16)

    M, N = x.shape
    packed_cols = (N + 1) // 2

    scales = torch.empty((M,), device=x.device, dtype=torch.float32)
    packed = torch.empty((M, packed_cols), device=x.device, dtype=torch.uint8)

    BLOCK_N = triton.next_power_of_2(N)
    assert BLOCK_N <= 4096, f"N={N} слишком велико для этого простого MVP; нужно tiled reduction"

    row_absmax_kernel[(M,)](
        x, scales,
        x.stride(0), x.stride(1),
        M, N,
        BLOCK_N=BLOCK_N,
    )

    BLOCK_PACKS = 256
    grid = (M, triton.cdiv(packed_cols, BLOCK_PACKS))
    quant_pack_int4_kernel[grid](
        x, scales, packed,
        x.stride(0), x.stride(1),
        packed.stride(0), packed.stride(1),
        M, N,
        BLOCK_PACKS=BLOCK_PACKS,
    )

    return packed, scales


In [3]:

def unpack_int4_packed_torch(packed: torch.Tensor, original_n: int) -> torch.Tensor:
    """
    packed: [M, ceil(N/2)] uint8
    return q: [M, N] int8 in range [-8, 7]
    """
    assert packed.dtype == torch.uint8

    low = (packed & 0x0F).to(torch.int16)
    high = ((packed >> 4) & 0x0F).to(torch.int16)

    q = torch.empty((packed.shape[0], packed.shape[1] * 2), device=packed.device, dtype=torch.int16)
    q[:, 0::2] = low
    q[:, 1::2] = high

    q = q[:, :original_n]
    q = (q - 8).to(torch.int8)
    return q


def dequantize_int4_packed_torch(
    packed: torch.Tensor,
    scales: torch.Tensor,
    original_n: int,
    out_dtype=torch.float16,
):
    q = unpack_int4_packed_torch(packed, original_n).to(torch.float32)
    x_hat = q * scales[:, None]
    return x_hat.to(out_dtype)


def tensor_nbytes(x: torch.Tensor) -> int:
    return x.numel() * x.element_size()


def pretty_mb(nbytes: int) -> str:
    return f"{nbytes / 1024**2:.3f} MiB"


## 1. Smoke test на маленькой матрице

Здесь проверяем, что:
- kernel запускается
- выход имеет правильную форму
- ошибка квантования разумная


In [4]:

torch.manual_seed(0)

M, N = 8, 16
W = torch.randn((M, N), device=device, dtype=torch.float16) * 0.7

packed, scales = quantize_fp16_to_int4_packed_triton(W)
W_hat = dequantize_int4_packed_torch(packed, scales, N, out_dtype=torch.float16)

print("W shape:", tuple(W.shape), W.dtype)
print("packed shape:", tuple(packed.shape), packed.dtype)
print("scales shape:", tuple(scales.shape), scales.dtype)

print("\nOriginal W[0, :8]:")
print(W[0, :8])

print("\nReconstructed W_hat[0, :8]:")
print(W_hat[0, :8])

abs_err = (W - W_hat).abs()
print("\nmean abs error:", abs_err.mean().item())
print("max abs error :", abs_err.max().item())


W shape: (8, 16) torch.float16
packed shape: (8, 8) torch.uint8
scales shape: (8,) torch.float32

Original W[0, :8]:
tensor([-0.6475, -0.2976, -1.8516,  0.1016, -0.0846, -0.4058, -0.4360, -0.2299],
       device='cuda:0', dtype=torch.float16)

Reconstructed W_hat[0, :8]:
tensor([-0.5288, -0.2644, -1.8516,  0.0000,  0.0000, -0.5288, -0.5288, -0.2644],
       device='cuda:0', dtype=torch.float16)

mean abs error: 0.048492431640625
max abs error : 0.1558837890625


## 2. Проверка, что память действительно уменьшается примерно в 4 раза

Сравниваем:
- хранение исходной матрицы `fp16`
- хранение `packed int4` (`uint8`) без учёта `scales`
- хранение `packed int4 + row scales`



In [5]:

M, N = 2048, 2048
W = torch.randn((M, N), device=device, dtype=torch.float16)

packed, scales = quantize_fp16_to_int4_packed_triton(W)

fp16_bytes = tensor_nbytes(W)
packed_only_bytes = tensor_nbytes(packed)
packed_plus_scales_bytes = tensor_nbytes(packed) + tensor_nbytes(scales)

print(f"fp16 storage:              {pretty_mb(fp16_bytes)}")
print(f"packed int4 only:          {pretty_mb(packed_only_bytes)}")
print(f"packed int4 + row scales:  {pretty_mb(packed_plus_scales_bytes)}")

print()
print(f"Compression (weights only):        {fp16_bytes / packed_only_bytes:.3f}x")
print(f"Compression (weights + scales):    {fp16_bytes / packed_plus_scales_bytes:.3f}x")


fp16 storage:              8.000 MiB
packed int4 only:          2.000 MiB
packed int4 + row scales:  2.008 MiB

Compression (weights only):        4.000x
Compression (weights + scales):    3.984x


## 3. Проверка reconstruction quality на более крупной матрице

Это не финальный метод для LLM inference, а только **первый sanity check**:
- после quantize+pack+unpack веса должны оставаться близки к исходным
- ошибка не должна быть “совсем плохой”


In [6]:

M, N = 1024, 1024
W = torch.randn((M, N), device=device, dtype=torch.float16) * 0.5

packed, scales = quantize_fp16_to_int4_packed_triton(W)
W_hat = dequantize_int4_packed_torch(packed, scales, N, out_dtype=torch.float16)

abs_err = (W - W_hat).abs()
rel_l2 = (W.float() - W_hat.float()).norm() / W.float().norm()

print("mean abs error:", abs_err.mean().item())
print("max abs error :", abs_err.max().item())
print("relative L2   :", rel_l2.item())


mean abs error: 0.0614013671875
max abs error : 0.171875
relative L2   : 0.14263342320919037


## 4. Мини-бенчмарк времени quantize+pack

Это не основной benchmark проекта, а просто доказательство, что базовая реализация реально запускается на GPU.


In [7]:

@torch.no_grad()
def benchmark_quant_pack(M=2048, N=2048, iters=50, warmup=10):
    W = torch.randn((M, N), device=device, dtype=torch.float16)

    for _ in range(warmup):
        packed, scales = quantize_fp16_to_int4_packed_triton(W)
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(iters):
        packed, scales = quantize_fp16_to_int4_packed_triton(W)
    torch.cuda.synchronize()
    t1 = time.perf_counter()

    avg_ms = (t1 - t0) * 1000 / iters
    return avg_ms

avg_ms = benchmark_quant_pack(M=2048, N=2048, iters=100, warmup=20)
print(f"Average quantize+pack time: {avg_ms:.3f} ms")


Average quantize+pack time: 0.045 ms


## 5. Короткий вывод

Тут мы демонстрируем :
- базовую Triton-реализацию **row-wise fp16 → int4 quantization**
- упаковку **2 x int4 → 1 byte**
- корректность 
- выигрыш по памяти порядка **4x по весам**

